In [1]:
import subprocess
import json
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import threading
import time
import os

# ─────────────────────────────────────────────────────────────
# 🧩 Utility Functions
# ─────────────────────────────────────────────────────────────

def get_namespaces():
    try:
        result = subprocess.run(
            ["kubectl", "get", "ns", "-o", "jsonpath={.items[*].metadata.name}"],
            capture_output=True, text=True, check=True
        )
        return result.stdout.strip().split()
    except subprocess.CalledProcessError:
        return []

def run_k8sgpt(namespace=None, explain=True):
    cmd = ["k8sgpt", "analyze", "--output", "json"]
    if explain:
        cmd.extend(["--explain", "--backend", "ollama"])
    if namespace:
        cmd.extend(["-n", namespace])

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        return json.loads(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"❌ Error: {e.stderr.strip()}")
        return None

def display_results(data, namespace):
    if not data or "results" not in data or not data["results"]:
        return widgets.HTML(f"<b>✅ No issues found in namespace <code>{namespace}</code>.</b>")

    df = pd.json_normalize(data["results"])
    for col in ["namespace", "reason", "explanation"]:
        if col not in df.columns:
            df[col] = "N/A"

    csv_filename = f"k8sgpt_{namespace}.csv"
    df.to_csv(csv_filename, index=False)

    display_items = [
        widgets.HTML(f"<b>🔍 Results for <code>{namespace}</code></b>"),
        widgets.Output(layout={"border": "1px solid gray"})
    ]
    with display_items[1]:
        display(df[["name", "kind", "namespace", "reason", "explanation"]])

    kind_count = df["kind"].value_counts().reset_index()
    kind_count.columns = ["Resource Kind", "Count"]
    fig = px.bar(kind_count, x="Resource Kind", y="Count", title=f"Issues in {namespace}", color="Count")

    chart_output = widgets.Output()
    with chart_output:
        fig.show()

    display_items.append(chart_output)
    display_items.append(widgets.HTML(f"📁 Auto-saved to <code>{csv_filename}</code>"))

    acc = widgets.Accordion(children=[widgets.VBox(display_items)])
    acc.set_title(0, f"Namespace: {namespace}")
    return acc, df

# ─────────────────────────────────────────────────────────────
# 🚀 Full Namespace Analyzer
# ─────────────────────────────────────────────────────────────

def analyze_all_namespaces(refresh_interval=None):
    clear_output()
    display(Markdown("## 🧠 K8sGPT Analysis In Progress..."))

    namespaces = get_namespaces()
    tab = widgets.Tab()
    tab_children = []
    all_dfs = []

    for ns in namespaces:
        data = run_k8sgpt(namespace=ns)
        widget, df = display_results(data, ns)
        tab_children.append(widget)
        all_dfs.append(df)

    tab.children = tab_children
    for i, ns in enumerate(namespaces):
        tab.set_title(i, ns)

    global_df = pd.concat(all_dfs, ignore_index=True)

    display(tab)
    display_summary(global_df)

    if refresh_interval:
        threading.Timer(refresh_interval * 60, lambda: analyze_all_namespaces(refresh_interval)).start()

# ─────────────────────────────────────────────────────────────
# 📊 Global Summary Chart
# ─────────────────────────────────────────────────────────────

def display_summary(global_df):
    display(Markdown("### 📈 Summary of Issues Across Namespaces"))
    grouped = global_df.groupby(["namespace", "kind"]).size().reset_index(name="Count")
    fig = px.sunburst(grouped, path=["namespace", "kind"], values="Count", title="Issues Breakdown")
    fig.show()

# ─────────────────────────────────────────────────────────────
# 📤 Upload and Visualize Existing Report
# ─────────────────────────────────────────────────────────────

def load_uploaded_json(change):
    clear_output()
    uploaded_file = change["new"]
    if uploaded_file:
        content = uploaded_file[list(uploaded_file.keys())[0]]["content"]
        data = json.loads(content.decode())
        ns = data.get("results", [{}])[0].get("namespace", "uploaded")
        display(Markdown("### 📂 Uploaded Report"))
        widget, _ = display_results(data, ns)
        display(widget)

upload_button = widgets.FileUpload(accept=".json", multiple=False)
upload_button.observe(load_uploaded_json, names="value")

# ─────────────────────────────────────────────────────────────
# 🎛️ Control Panel
# ─────────────────────────────────────────────────────────────

refresh_input = widgets.IntText(value=0, description="Auto-Refresh (min):", min=0)
run_button = widgets.Button(description="🚀 Run Full Analysis")

def on_run(b):
    interval = refresh_input.value
    analyze_all_namespaces(refresh_interval=interval if interval > 0 else None)

run_button.on_click(on_run)

display(Markdown("## 🧪 K8sGPT Analyzer Suite"))
display(widgets.HBox([refresh_input, run_button]))
display(Markdown("### 📤 Or Load Existing Report"))
display(upload_button)


## 🧪 K8sGPT Analyzer Suite

### 📤 Or Load Existing Report

FileUpload(value=(), accept='.json', description='Upload')